# Macroeconomic Stress — One-Year-Ahead Evaluation

This notebook tests whether **prior-year indicators** predict next-year rule-defined stress. It does not reproduce or validate the near-perfect same-year benchmark. Configure your own project/table; do not commit cloud identifiers or credentials.

In [ ]:
# Run in Google Colab; authenticate with the account authorized to read your table.
from google.colab import auth
auth.authenticate_user()
from google.cloud import bigquery
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

## Configure data source

**Keep your real cloud ID out of the GitHub copy.** Modify the cell in your private Colab copy or enter it when prompted. This notebook only reads the existing table.

In [ ]:
PROJECT_ID = input("Your GCP project ID (not saved in this notebook): ").strip()
TABLE_ID = input("Dataset.table (default macro_ml_project.macro_stress_ready): ").strip() or "macro_ml_project.macro_stress_ready"
if not PROJECT_ID or not TABLE_ID.replace("_", "").replace(".", "").isalnum() or TABLE_ID.count(".") != 1:
    raise ValueError("Provide a project ID and dataset.table")
client = bigquery.Client(project=PROJECT_ID)
query = f"SELECT * FROM `{PROJECT_ID}.{TABLE_ID}`"
df = client.query(query).to_dataframe()
print("Rows, columns:", df.shape)
print("Years:", df.year.min(), df.year.max())
print("Stress class counts:\n", df.economic_stress.value_counts(dropna=False))

## Data checks and one-year lags

Lags use consecutive years only; missing predictor values are imputed **inside each fitted pipeline**. World Bank aggregates should be filtered from the source table. The economic stress label must already exist in the BQ table.

In [ ]:
required = ["country_code", "year", "economic_stress"]
base_features = [
  "gdp_growth_pct", "gdp_per_capita_current_usd", "inflation_pct", "unemployment_pct",
  "current_account_pct_gdp", "exports_pct_gdp", "imports_pct_gdp", "trade_pct_gdp",
  "fdi_net_inflows_pct_gdp", "exchange_rate_lcu_per_usd", "population",
  "misery_index", "trade_balance_proxy", "growth_inflation_gap",
  "external_pressure_score", "exchange_rate_change_pct"
]
missing_columns = [c for c in required + base_features if c not in df.columns]
if missing_columns: raise ValueError(f"Missing columns: {missing_columns}")
if df.duplicated(["country_code", "year"]).any():
    raise ValueError("Duplicate country-year observations: deduplicate before analysis")
df = df.copy()
df["year"] = pd.to_numeric(df["year"], errors="raise").astype(int)
for col in base_features:
    df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
df = df.sort_values(["country_code", "year"])
for col in base_features:
    df[col + "_lag1"] = df.groupby("country_code")[col].shift(1)
df["prev_year"] = df.groupby("country_code")["year"].shift(1)
lagged = df.loc[(df.year - df.prev_year == 1) & df.economic_stress.isin([0, 1])].copy()
lagged_cols = [c + "_lag1" for c in base_features]
lagged = lagged.loc[lagged[lagged_cols].notna().any(axis=1)].copy()
print("Model observations",len(lagged),"missing rates",lagged[lagged_cols].isna().mean().sort_values(ascending=False).head())

## Chronological holdout

Targets for 1996–2016 are eligible for training; 2017–2020 for testing. No random splitting across years. Model tuning, if done, must occur within the training period only.

In [ ]:
train = lagged.loc[lagged.year <= 2016]
test = lagged.loc[(lagged.year >= 2017) & (lagged.year <= 2020)]
if train.empty or test.empty: raise ValueError("Insufficient rows for the proposed chronological split")
X_train, y_train = train[lagged_cols], train.economic_stress.astype(int)
X_test, y_test = test[lagged_cols], test.economic_stress.astype(int)
if y_train.nunique() < 2: raise ValueError("Training period contains one class")
print("Training:",len(train),"Test:",len(test),"Positive test share:",round(y_test.mean(),3))

## Fit model benchmarks on t−1 variables and report honest holdout performance

In [ ]:
models = {
 "Most frequent": Pipeline([("imputer",SimpleImputer(strategy="median",keep_empty_features=True)),("model",DummyClassifier(strategy="most_frequent"))]),
 "Logistic Regression": Pipeline([("imputer",SimpleImputer(strategy="median",keep_empty_features=True)),("scale",StandardScaler()),("model",LogisticRegression(max_iter=2000,class_weight="balanced",random_state=42))]),
 "Decision Tree": Pipeline([("imputer",SimpleImputer(strategy="median",keep_empty_features=True)),("model",DecisionTreeClassifier(max_depth=5,class_weight="balanced",random_state=42))]),
 "Random Forest": Pipeline([("imputer",SimpleImputer(strategy="median",keep_empty_features=True)),("model",RandomForestClassifier(n_estimators=300,class_weight="balanced",random_state=42,n_jobs=-1))]),
 "Gradient Boosting": Pipeline([("imputer",SimpleImputer(strategy="median",keep_empty_features=True)),("model",GradientBoostingClassifier(random_state=42))])
}
rows = []
for name, model in models.items():
    model.fit(X_train,y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1] if hasattr(model,"predict_proba") else None
    tn,fp,fn,tp=confusion_matrix(y_test,pred,labels=[0,1]).ravel()
    rows.append({"model":name,"accuracy":accuracy_score(y_test,pred),"precision":precision_score(y_test,pred,zero_division=0),"recall":recall_score(y_test,pred,zero_division=0),"f1":f1_score(y_test,pred,zero_division=0),"roc_auc":roc_auc_score(y_test,proba) if proba is not None and y_test.nunique()==2 else np.nan,"TN":tn,"FP":fp,"FN":fn,"TP":tp})
results = pd.DataFrame(rows).sort_values("f1",ascending=False)
display(results.round(4))

## Interpret results

1. Compare models to the most-frequent baseline.
2. Note operational false alarms = FP / (FP + TN), **not the far-right endpoint of an ROC curve**.
3. These labels are threshold-defined stress, not independently confirmed crises.
4. Results are out-of-time but still require independent external validation.
5. Never overwrite the original project report's metrics with these results without describing the changed task.

In [ ]:
results["false_positive_rate"] = results.FP / (results.FP + results.TN).replace(0,np.nan)
display(results[["model","f1","recall","precision","false_positive_rate","roc_auc"]].round(4))
# Optional local download (not included in GitHub ZIP):
# results.to_csv("forward_warning_holdout_results.csv", index=False)